# Does Adaptive Conformal Recover Faster After a Regime Shift?

This ties our two findings together: we know coverage depends on volatility
regime, and we know adaptive conformal adjusts itself over time. Here we
check whether adaptive conformal specifically helps right after the market
switches from one regime to another, compared to fixed conformal.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Load everything: forecasts, actuals, regime labels for Sensex

In [2]:
forecasts = np.load("../data/forecasts.npy")
actuals = np.load("../data/actuals.npy")
regimes = np.load("../data/sensex_regime_labels.npy", allow_pickle=True)

day = 15
quantile_levels = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

day_forecasts = forecasts[:, day, :]
day_actuals = actuals[:, day]

lower = day_forecasts[:, 1]
upper = day_forecasts[:, 9]

## Same time split as before: 60% calibration, 40% test

In [3]:
n = len(day_actuals)
split_point = int(n * 0.6)

cal_lower, test_lower = lower[:split_point], lower[split_point:]
cal_upper, test_upper = upper[:split_point], upper[split_point:]
cal_actuals, test_actuals = day_actuals[:split_point], day_actuals[split_point:]
test_regimes = regimes[split_point:]

len(test_actuals), len(test_regimes)

(141, 141)

## Recompute fixed conformal coverage, point by point (not just the average)

In [4]:
cal_scores = np.maximum(cal_lower - cal_actuals, cal_actuals - cal_upper)
q_hat = np.quantile(cal_scores, 0.8)

fixed_lower = test_lower - q_hat
fixed_upper = test_upper + q_hat

fixed_covered = (test_actuals >= fixed_lower) & (test_actuals <= fixed_upper)
fixed_covered = fixed_covered.astype(int)

## Recompute adaptive conformal coverage, point by point (same method as before)

In [5]:
target_alpha = 0.2
gamma = 0.01
alpha_t = target_alpha
score_history = list(cal_scores)

adaptive_covered = []

for i in range(len(test_actuals)):
    alpha_t = np.clip(alpha_t, 0.01, 0.5)
    q_t = np.quantile(score_history, 1 - alpha_t)

    lo = test_lower[i] - q_t
    hi = test_upper[i] + q_t

    actual = test_actuals[i]
    covered = (actual >= lo) and (actual <= hi)
    adaptive_covered.append(int(covered))

    err_t = 0 if covered else 1
    alpha_t = alpha_t + gamma * (target_alpha - err_t)

    this_score = max(test_lower[i] - actual, actual - test_upper[i])
    score_history.append(this_score)

adaptive_covered = np.array(adaptive_covered)

## Find regime shift points in the test set

A "shift point" is where the regime label changes from the previous window
to the current one.

In [6]:
shift_points = []
for i in range(1, len(test_regimes)):
    if test_regimes[i] != test_regimes[i-1]:
        shift_points.append(i)

print("number of shift points found:", len(shift_points))
print(shift_points)

number of shift points found: 26
[9, 10, 12, 13, 21, 33, 49, 52, 57, 61, 62, 87, 89, 96, 98, 100, 102, 104, 105, 108, 110, 117, 121, 137, 139, 140]


## Compare coverage in the window right after a shift

For each shift point, look at the next 5 windows and check what fraction
were covered, for fixed vs adaptive. Average this across all shift points.

In [7]:
recovery_window = 5

fixed_recovery_scores = []
adaptive_recovery_scores = []

for sp in shift_points:
    end = min(sp + recovery_window, len(test_actuals))
    fixed_recovery_scores.append(fixed_covered[sp:end].mean())
    adaptive_recovery_scores.append(adaptive_covered[sp:end].mean())

fixed_recovery = np.mean(fixed_recovery_scores)
adaptive_recovery = np.mean(adaptive_recovery_scores)

print("fixed conformal coverage right after a regime shift:", round(fixed_recovery * 100, 1), "%")
print("adaptive conformal coverage right after a regime shift:", round(adaptive_recovery * 100, 1), "%")

fixed conformal coverage right after a regime shift: 80.0 %
adaptive conformal coverage right after a regime shift: 80.0 %


## Compare to overall coverage (not just after shifts) as a baseline

In [8]:
print("fixed conformal overall coverage:", round(fixed_covered.mean() * 100, 1), "%")
print("adaptive conformal overall coverage:", round(adaptive_covered.mean() * 100, 1), "%")

fixed conformal overall coverage: 78.0 %
adaptive conformal overall coverage: 78.7 %


## How to read this

If adaptive conformal's coverage right after a shift is closer to its
overall average than fixed conformal's is, that means adaptive conformal
handles regime shifts better, recovering faster instead of staying wrong
for a while.

If both methods drop by a similar amount right after a shift, that means
adaptive conformal isn't actually helping with shifts specifically, even
though it helped overall.

**Regime shift recovery (null result):** we tested whether adaptive conformal
prediction recovers coverage faster than fixed conformal prediction in the
windows immediately following a volatility regime shift. Both methods showed
identical coverage (80.0%) in the 5 windows following a shift, despite
adaptive conformal's modest overall advantage (78.7% vs 78.0%). We attribute
this null result partly to the frequency of detected shift points (26 shifts
across 140 test windows) and their tendency to cluster closely in time
(e.g., shifts at windows 9, 10, 12, 13), which likely reflects boundary-
crossing noise from the median-split regime definition rather than clean,
isolated transitions between stable regimes. A recovery-speed analysis
would likely require a stricter regime definition (e.g., requiring a
minimum regime duration before and after a transition) to isolate genuinely
distinct shifts.